## Agregaciones con MapReduce
Tres vistas sobre los pedidos (se excluyen los cancelados en ventas y productos vendidos):

| Vista | Qué responde | Reduce |
|---|---|---|
| `ventas_por_categoria` | Ventas totales por categoría (usa el *snapshot* de cada línea) | `_sum` |
| `ticket_por_estado` | Cantidad, suma, mínimo y máximo de pedidos → ticket promedio | `_stats` |
| `productos_mas_vendidos` | Unidades vendidas por producto | `_sum` |

La primera consulta construye la vista y puede tardar un poco.

In [ ]:
def crear_vistas_agregacion():
    guardar_design_doc({
        "_id": "_design/agregaciones",
        "views": {
            "ventas_por_categoria": {
                "map": (
                    "function (doc) { if (doc.type === 'order' && doc.estado !== 'cancelado') {"
                    " doc.lineas_detalle.forEach(function (l) { emit(l.categoria_snapshot, l.subtotal); }); } }"
                ),
                "reduce": "_sum",
            },
            "ticket_por_estado": {
                "map": "function (doc) { if (doc.type === 'order') { emit(doc.estado, doc.monto_total); } }",
                "reduce": "_stats",
            },
            "productos_mas_vendidos": {
                "map": (
                    "function (doc) { if (doc.type === 'order' && doc.estado !== 'cancelado') {"
                    " doc.lineas_detalle.forEach(function (l) { emit(l.producto_id, l.cantidad); }); } }"
                ),
                "reduce": "_sum",
            },
        },
    })


def consultar_vista(nombre):
    url = f"{DB_URL}/_design/agregaciones/_view/{nombre}?group=true"
    respuesta = SESSION.get(url, timeout=600)
    respuesta.raise_for_status()
    return respuesta.json().get("rows", [])


def ejecutar_agregaciones():
    crear_vistas_agregacion()

    print("--- Ventas por categoría ---")
    ventas = sorted(consultar_vista("ventas_por_categoria"), key=lambda r: r["value"], reverse=True)
    for r in ventas:
        print(f"  {str(r['key']):<18} ₡{r['value']:>14,.2f}")

    print("\n--- Ticket promedio por estado de pedido ---")
    for r in consultar_vista("ticket_por_estado"):
        v = r["value"]
        print(f"  {r['key']:<12} pedidos: {v['count']:>6} | promedio: ₡{v['sum'] / v['count']:>9,.2f} "
              f"| mín: ₡{v['min']:,.2f} | máx: ₡{v['max']:,.2f}")

    print("\n--- Top 5 productos más vendidos (unidades) ---")
    top = sorted(consultar_vista("productos_mas_vendidos"), key=lambda r: r["value"], reverse=True)[:5]
    for r in top:
        print(f"  {r['key']}  {r['value']} unidades")

    registrar("Agregaciones MapReduce", bool(ventas and top), "3 vistas consultadas")


ejecutar_agregaciones()

--- Ventas por categoría ---
  Artesanias         $ 15,989,377.14
  Ropa               $ 15,857,943.45
  Hogar              $ 15,617,635.94
  Tecnologia         $ 15,444,370.03
  Cuidado_Personal   $ 15,376,700.93
  Libros             $ 15,356,986.53
  Alimentos          $ 15,080,301.64
  Mascotas           $ 15,034,859.26

--- Ticket promedio por estado de pedido ---
  cancelado    pedidos:  16827 | promedio: $ 3,770.31 | mín: $7.18 | máx: $14,262.16
  completado   pedidos:  16690 | promedio: $ 3,755.05 | mín: $5.70 | máx: $15,846.64
  pendiente    pedidos:  16483 | promedio: $ 3,706.03 | mín: $5.14 | máx: $15,591.81

--- Top 5 productos más vendidos (unidades) ---
  PROD-013242  28 unidades
  PROD-021866  28 unidades
  PROD-014110  26 unidades
  PROD-016627  26 unidades
  PROD-003762  25 unidades
  [OK] Req 6: agregaciones MapReduce  3 vistas consultadas


## Evolución del esquema sin migración
Se demuestra con tres acciones sobre la misma base de datos, sin `ALTER TABLE` ni migración:
1. Una **categoría nueva** con atributos nunca vistos.
2. Un **atributo nuevo** en un producto existente.
3. Una **validación** (`validate_doc_update`) que mantiene la consistencia: rechaza documentos inválidos aunque el esquema sea flexible.

In [ ]:
def instalar_validacion():
    guardar_design_doc({
        "_id": "_design/validacion",
        "validate_doc_update": (
            "function (newDoc, oldDoc, userCtx) {"
            "  if (newDoc._deleted || newDoc._id.indexOf('_design/') === 0) { return; }"
            "  if (['product', 'customer', 'order'].indexOf(newDoc.type) === -1) {"
            "    throw ({forbidden: 'tipo de documento inválido'});"
            "  }"
            "  if (newDoc.type === 'product') {"
            "    if (!newDoc.categoria) { throw ({forbidden: 'categoría requerida'}); }"
            "    if (typeof newDoc.precio !== 'number' || newDoc.precio < 0) {"
            "      throw ({forbidden: 'precio inválido'});"
            "    }"
            "  }"
            "  if (newDoc.type === 'order' && "
            "      (!newDoc.lineas_detalle || newDoc.lineas_detalle.length === 0)) {"
            "    throw ({forbidden: 'el pedido necesita líneas'});"
            "  }"
            "}"
        )
    })


def demostrar_evolucion_esquema():
    instalar_validacion()
    print("[1] Validación instalada (consistencia)\n")

    # A) Categoría completamente nueva, con sus propios atributos
    nuevo = {
        "_id": "PROD-999999", "type": "product",
        "nombre": "Drone de Monitoreo Agrícola", "categoria": "Tecnologia_Agricola",
        "precio": 450000, "stock": 10,
        "atributos": {"autonomia_minutos": 45, "camara_multiespectral": True, "alcance_km": 12.5},
    }
    r = SESSION.put(f"{DB_URL}/PROD-999999", json=nuevo, timeout=30)
    estado = {201: "creada", 202: "creada", 409: "ya existía"}.get(r.status_code, f"error {r.status_code}")
    print(f"[2] Categoría nueva 'Tecnologia_Agricola': {estado}")
    hallados = buscar({"type": "product", "categoria": "Tecnologia_Agricola"},
                      use_index="idx_product_category", titulo="    consulta de la categoría nueva")

    # B) Atributo nuevo en un producto existente
    ropa = SESSION.post(f"{DB_URL}/_find", timeout=60,
                        json={"selector": {"type": "product", "categoria": "Ropa"}, "limit": 1}).json()["docs"][0]
    ropa["atributos"]["certificacion_ecologica"] = True
    r2 = SESSION.put(f"{DB_URL}/{ropa['_id']}", json=ropa, timeout=30)
    print(f"[3] Atributo nuevo 'certificacion_ecologica' en {ropa['_id']}: HTTP {r2.status_code}")

    # C) Consistencia: un producto inválido es rechazado por la base de datos
    invalido = {"type": "product", "categoria": "Ropa", "nombre": "Prueba", "precio": -5}
    r3 = SESSION.post(DB_URL, json=invalido, timeout=30)
    print(f"[4] Producto con precio negativo: HTTP {r3.status_code} -> {r3.json().get('reason')}")

    registrar("Evolución del esquema",
              r.status_code in (201, 202, 409) and bool(hallados.get("docs")) and r3.status_code == 403 and r2.status_code in (201, 202),
              "categoría nueva, atributo nuevo y validación")


demostrar_evolucion_esquema()

[1] Validación instalada (consistencia)

[2] Categoría nueva 'Tecnologia_Agricola': creada
    consulta de la categoría nueva: 1 resultados | docs examinados: 1 | 16.0 ms
   aviso de CouchDB: _design/idx_product_category was not used because it does not contain a valid index for this query
[3] Atributo nuevo 'certificacion_ecologica' en PROD-000001: HTTP 201
[4] Producto con precio negativo: HTTP 403 -> precio invalido
  [OK] Req 7: evolución del esquema  categoría nueva, atributo nuevo y validación


## Resumen de pruebas

In [ ]:
print("RESUMEN DE PRUEBAS - MERCADOTICO 360")
for prueba, (ok, detalle) in RESULTADOS.items():
    estado = "Correcto" if ok else "Revisar"
    print(f"{estado}: {prueba} - {detalle}")
print(
    f"\nPruebas correctas: "
    f"{sum(ok for ok, _ in RESULTADOS.values())}/{len(RESULTADOS)}"
)

=== RESUMEN DE REQUISITOS · MERCADOTICO 360 ===
[OK] Req 1-2: diseño y atributos variables  
[OK] Req 3: índices  4 índices
[OK] Req 4: antes/después de índices  27.3x menos documentos examinados
[OK] Req 4: categoría + precio + atributo  611 productos
[OK] Req 5: pedidos y actualización  11 pedidos recuperados
[OK] Req 6: agregaciones MapReduce  3 vistas consultadas
[OK] Req 7: evolución del esquema  categoría nueva, atributo nuevo y validación

7/7 requisitos verificados
